# Lab: Private Set Intersection
**Cloud Encryption, Day III, Session III**

Agency A (Alice) holds the codici fiscali $X$ of 100 benefit recipients; Agency B (the Hatter) holds the codici fiscali $Y$ of 1000 taxpayers with income above the threshold. All identifiers are **synthetic**.

* Every exercise has cells marked `# TODO`: write your code there, then run the **check** cell below it.
* Run the next two cells first: the helper library (group, hash to the group, the agencies) and the data.
* **Toy parameters**: nothing here is secure. Group elements are Python integers mod `p`; `pow(a, k, p)` is $a^k$, `pow(r, -1, q)` is $r^{-1}\bmod q$.

In [ ]:
# ===== Helper library (run me first) =====
"""Helper library for the Day III lab (Private Set Intersection).
Toy parameters, NOT secure: for understanding only.  Pure Python, standard library only.
Runs in plain Jupyter and in JupyterLite (Pyodide kernel)."""
import hashlib, random, string, time, sys

rng = random.Random(2026)    # fixed seed: everybody gets the same numbers
                             # (a real implementation draws keys with the `secrets` module)

# ---- the group (toy size) ------------------------------------------------------
# RFC 2409 (Oakley group 2): p = 2q + 1 is a 1024-bit safe prime.
P_HEX = ("FFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD129024E088A67CC74"
         "020BBEA63B139B22514A08798E3404DDEF9519B3CD3A431B302B0A6DF25F1437"
         "4FE1356D6D51C245E485B576625E7EC6F44C42E9A637ED6B0BFF5CB6F406B7ED"
         "EE386BFB5A899FA5AE9F24117C4B1FE649286651ECE65381FFFFFFFFFFFFFFFF")
p = int(P_HEX, 16)
q = (p - 1) // 2             # prime: the order of the group G of squares mod p
g = 4                        # 4 = 2^2 generates G

def rand_exp():
    """A uniform exponent in Z_q^* = {1, ..., q-1}."""
    return rng.randrange(1, q)

def hash_to_group(x: str) -> int:
    """H: strings -> G. Expand SHA256 to 2304 bits, reduce mod p, square (squares lie in G)."""
    out = b"".join(hashlib.sha256(bytes([i]) + x.encode()).digest() for i in range(9))
    return pow(int.from_bytes(out, "big") % p, 2, p)

def sha(x: str) -> str:
    """SHA256 of a string, as hex."""
    return hashlib.sha256(x.encode()).hexdigest()

# ---- the two agencies (synthetic data) -------------------------------------------
LETTERS = string.ascii_uppercase
MONTHS = "ABCDEHLMPRST"
def fake_cf(r):
    """A synthetic identifier shaped like a codice fiscale (format only, no check character)."""
    s = "".join(r.choice(LETTERS) for _ in range(6))
    s += f"{r.randint(40, 99):02d}" + r.choice(MONTHS) + f"{r.choice([r.randint(1, 31), r.randint(41, 71)]):02d}"
    s += r.choice("ABCDEFGHILM") + f"{r.randint(1, 999):03d}" + r.choice(LETTERS)
    return s

def make_agencies(seed=1, registry_size=200_000, n=100, m=1000, overlap=37):
    """registry (what Agency B can enumerate), X (Agency A, n ids), Y (Agency B, m ids), |X & Y| = overlap."""
    r = random.Random(seed)
    registry = list(dict.fromkeys(fake_cf(r) for _ in range(registry_size)))
    r.shuffle(registry)
    common = registry[:overlap]
    X = common + registry[overlap:n]
    Y = common + registry[n:n + m - overlap]
    r.shuffle(X); r.shuffle(Y)
    return registry, X, Y

def excess_income(Y, seed=3):
    """Agency B's value per taxpayer: declared income above the threshold, in euros."""
    r = random.Random(seed)
    return {y: r.randint(500, 40_000) for y in Y}

# ---- OT: PVW in messy mode (DDH) ---------------------------------------------------
def pvw_setup_messy():
    """crs = (g0, h0, g1, h1) with h_b = g_b^{x_b}, x0 != x1 nonzero; trapdoor (x0, x1)."""
    g0, g1 = hash_to_group("crs g0"), hash_to_group("crs g1")
    x0 = rand_exp(); x1 = rand_exp()
    while x1 == x0: x1 = rand_exp()
    return (g0, pow(g0, x0, p), g1, pow(g1, x1, p)), (x0, x1)

def pvw_keygen(crs, sigma):
    """Key for branch sigma: pk = (g_sigma^r, h_sigma^r), sk = r."""
    g0, h0, g1, h1 = crs
    r = rand_exp()
    gs, hs = (g0, h0) if sigma == 0 else (g1, h1)
    return (pow(gs, r, p), pow(hs, r, p)), r

def pvw_dec(r, c):
    """v / u^r."""
    u, v = c
    return v * pow(pow(u, r, p), -1, p) % p

# ---- OT extension ----------------------------------------------------------------
LAM = 128
def H(j: int, v: int, ell: int = 128) -> int:
    """The hash H(j, v) of IKNP, on a 128-bit integer v, with ell output bits."""
    d = hashlib.sha256(j.to_bytes(4, "big") + v.to_bytes(LAM // 8, "big")).digest()
    return int.from_bytes(d, "big") >> (256 - ell)

def rows(cols, N):
    """Transpose: cols = 128 integers of N bits (the columns); returns N integers of 128 bits (the rows)."""
    out = [0] * N
    for i, c in enumerate(cols):
        j = 0
        while c:
            if c & 1: out[j] |= 1 << i
            c >>= 1; j += 1
    return out

# ---- hashing ---------------------------------------------------------------------
def two_core(n_items, n_pos, k=3, r=None):
    """Number of keys left in the 2-core after peeling a random k-hash OKVS with n_pos positions (0 = success)."""
    r = r or rng
    edges = [r.sample(range(n_pos), k) for _ in range(n_items)]
    deg = [0] * n_pos; inc = [[] for _ in range(n_pos)]
    for e, vs in enumerate(edges):
        for v in vs: deg[v] += 1; inc[v].append(e)
    alive = [True] * n_items
    stack = [v for v in range(n_pos) if deg[v] == 1]
    while stack:
        v = stack.pop()
        if deg[v] != 1: continue
        e = next(e for e in inc[v] if alive[e]); alive[e] = False
        for u in edges[e]:
            deg[u] -= 1
            if deg[u] == 1: stack.append(u)
    return sum(alive)

# ---- Paillier (toy, 512-bit modulus) -------------------------------------------------
def _is_prime(x):
    if x < 2: return False
    for s in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29):
        if x % s == 0: return x == s
    d, t = x - 1, 0
    while d % 2 == 0: d //= 2; t += 1
    for a in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        y = pow(a, d, x)
        if y in (1, x - 1): continue
        for _ in range(t - 1):
            y = y * y % x
            if y == x - 1: break
        else: return False
    return True

def paillier_keygen(bits=512):
    """(N, sk): N = p'q' with two random primes of bits/2 bits."""
    def prime(b):
        while True:
            x = rng.getrandbits(b) | (1 << (b - 1)) | 1
            if _is_prime(x): return x
    a, b = prime(bits // 2), prime(bits // 2)
    N = a * b; lam = (a - 1) * (b - 1)
    return N, (lam, pow(lam, -1, N))

def paillier_enc(N, msg):
    """Enc(msg) = (1+N)^msg * r^N mod N^2."""
    r = rng.randrange(1, N)
    return pow(N + 1, msg, N * N) * pow(r, N, N * N) % (N * N)

def paillier_add(N, c1, c2):
    """Enc(m1) * Enc(m2) = Enc(m1 + m2)."""
    return c1 * c2 % (N * N)

def paillier_dec(N, sk, c):
    lam, mu = sk
    return (pow(c, lam, N * N) - 1) // N * mu % N

def selftest():
    assert pow(g, q, p) == 1 and pow(hash_to_group("test"), q, p) == 1 and _is_prime(q)
    N, sk = paillier_keygen(128); assert paillier_dec(N, sk, paillier_add(N, paillier_enc(N, 20), paillier_enc(N, 22))) == 42
    print("Library OK: 1024-bit group (RFC 2409), q prime, g = 4")
    t0 = time.time(); [pow(g, rand_exp(), p) for _ in range(20)]
    print(f"(one exponentiation here: {(time.time() - t0) / 20 * 1000:.1f} ms; Python {sys.version.split()[0]})")


selftest()

In [ ]:
registry, X, Y = make_agencies()
print(len(registry), "identifiers in the registry;", "|X| =", len(X), " |Y| =", len(Y), " |X & Y| =", len(set(X) & set(Y)))
print("example:", X[0])

## Exercise 1 — Break hashed PSI
Agency A sends $\mathrm{SHA256}(x)$ for every $x\in X$. Play Agency B: hash the whole `registry` into a dictionary (hash $\to$ identifier) and look up every received hash.

In [ ]:
def hashed_psi_attack(registry, published):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
published = [sha(x) for x in X]                       # what Agency A sends
t0 = time.time(); found = hashed_psi_attack(registry, published); dt = time.time() - t0
print(f"Agency B recovers {len(found)} of {len(X)} identifiers (not only the {len(set(X) & set(Y))} in common)")
print(f"{dt:.2f} s for {len(registry)} identifiers; for all 6e7 codici fiscali about {dt * 6e7 / len(registry):.0f} s")

**Think:** (a) Does a public salt help? (b) A secret key? Who would hold it? (c) Which property of a PRF is missing here?

## Exercise 2 — DH-OPRF and DH-PSI
* `blind(x)`: pick $r\in\mathbb Z_q^*$ (`rand_exp()`), return $(r,\ a=H(x)^r)$.
* `evaluate(k, a)`: the Hatter returns $a^k$.
* `unblind(r, b)`: return $b^{1/r}$, with $1/r$ computed mod $q$.
* `dh_psi(X, Y)`: blind every $x$; the Hatter evaluates and also sends $\{H(y)^k\}_{y\in Y}$; unblind and output the $x$ whose value is in the list.

In [ ]:
def blind(x):
    raise NotImplementedError  # TODO

def evaluate(k, a):
    raise NotImplementedError  # TODO

def unblind(r, b):
    raise NotImplementedError  # TODO

def dh_psi(X, Y):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
t0 = time.time(); res = dh_psi(X, Y); dt = time.time() - t0
print(f"DH-PSI: {len(res)} matches, correct: {set(res) == set(X) & set(Y)}; {dt:.1f} s")
print(f"exponentiations: Alice {2 * len(X)}, Hatter {len(X) + len(Y)}")

**Think:** Which values does the Hatter see? Why do they look random whatever $x$ is?

## Exercise 3 — PSI-CA, and what still leaks
* `dh_psi_ca(X, Y)`: Alice sends $\{H(x)^\alpha\}$; the Hatter raises them to $\beta$ and **shuffles** (`rng.shuffle`); he also sends $\{H(y)^\beta\}$; Alice raises those to $\alpha$ and counts the matches.
* `membership(X, Y, target)`: as Alice, run PSI-CA twice, with $X$ and with $X\setminus\{\mathit{target}\}$, and decide whether `target` is in $Y$.

In [ ]:
def dh_psi_ca(X, Y):
    raise NotImplementedError  # TODO

def membership(X, Y, target):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
print("PSI-CA:", dh_psi_ca(X, Y))                     # three PSI-CA runs in this cell: about 30 s in the browser
target = sorted(set(X) & set(Y))[0]
print(f"is {target} in Y? membership says {membership(X, Y, target)}")

**Think:** (a) What does Alice learn in one run? And the Hatter? (b) Can any protocol for PSI-CA stop the attack of `membership`?

## Exercise 4 — Oblivious transfer: passive, then messy
**(a)** `elgamal_ot(M0, M1, sigma)`: Alice sets $h_\sigma=g^x$ and $h_{1-\sigma}$ = `hash_to_group` of a random string (nobody knows its discrete log); the Hatter sends $c_b=(g^{r_b},\ h_b^{r_b}\cdot M_b)$ for $b=0,1$; Alice decrypts $c_\sigma$. Then `cheating_receiver(M0, M1)`: Alice picks **both** keys as $g^{x_0}, g^{x_1}$.

**(b)** PVW in messy mode (the library has `pvw_setup_messy`, `pvw_keygen`, `pvw_dec`). Write `pvw_enc(crs, pk, b, M)` $=(g_b^sh_b^t,\ g^sh^t\cdot M)$ for $pk=(g,h)$, returning `None` if $g=1$, and `find_messy(trap, pk)`: $0$ if $h\neq g^{x_0}$, else $1$.

In [ ]:
def elgamal_ot(M0, M1, sigma):
    raise NotImplementedError  # TODO

def cheating_receiver(M0, M1):
    raise NotImplementedError  # TODO

def pvw_enc(crs, pk, b, M):
    raise NotImplementedError  # TODO

def find_messy(trap, pk):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
M0, M1 = hash_to_group("message 0"), hash_to_group("message 1")
print("ElGamal OT:", elgamal_ot(M0, M1, 0) == M0, elgamal_ot(M0, M1, 1) == M1)
print("cheating receiver gets both:", cheating_receiver(M0, M1) == [M0, M1])
crs, trap = pvw_setup_messy()
pk, sk = pvw_keygen(crs, 1)
print("PVW: branch 1 decrypts:", pvw_dec(sk, pvw_enc(crs, pk, 1, M1)) == M1,
      "| find_messy:", find_messy(trap, pk),
      "| branch 0 decrypts:", pvw_dec(sk, pvw_enc(crs, pk, 0, M0)) == M0)

**Think:** (a) What must a simulator do against `cheating_receiver`? (b) Try a malformed key $(g,h)$ of your choice in PVW: can you make both branches decrypt?

## Exercise 5 — A toy IKNP
`iknp(pairs, bits)`: $N$ OTs on 128-bit strings from $\lambda=128$ base OTs (simulated ideally). Bit vectors are Python integers: bit $j$ of `c` is `(c >> j) & 1`.
* Alice: random columns $r_i$ of $N$ bits (`rng.getrandbits(N)`), $i<128$; the Hatter: $\Delta$ of 128 bits.
* Base OTs: the Hatter gets $t_i=r_i$ if $\Delta_i=0$, else $r_i\oplus b$ ($b$ = Alice's bits as an integer).
* Rows: `rows(cols, N)` (library). The Hatter sends $y_0^j=H(j,t^j)\oplus M_0^j$ and $y_1^j=H(j,t^j\oplus\Delta)\oplus M_1^j$; Alice outputs $y^j_{b_j}\oplus H(j,r^j)$.

In [ ]:
def iknp(pairs, bits):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
N = 10_000
pairs = [(rng.getrandbits(128), rng.getrandbits(128)) for _ in range(N)]
bits = [rng.getrandbits(1) for _ in range(N)]
t0 = time.time(); res = iknp(pairs, bits); dt = time.time() - t0
print(f"IKNP: {N} OTs, correct: {res == [pairs[j][bits[j]] for j in range(N)]}; {dt:.1f} s, {dt / N * 1e6:.0f} microseconds per OT")
t0 = time.time(); elgamal_ot(M0, M1, 0); print(f"one ElGamal OT: {(time.time() - t0) * 1e3:.0f} ms")

**Think:** What could a malicious Alice do by using a different $b$ in some columns?

## Exercise 6 — Cuckoo hashing and peeling
`cuckoo_insert(n_items, n_bins, k, r)`: every item gets `k` random bins (`r.randrange(n_bins)`); insert items one by one, evicting the occupant to its next bin; return `False` after 500 evictions in a row, `True` if all items are placed. The check cell measures success rates; `two_core` (library) does the same for peeling.

In [ ]:
def cuckoo_insert(n_items, n_bins, k, r):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
r = random.Random(5); n_bins = 1000
for k, loads in ((2, (0.40, 0.45, 0.50, 0.55)), (3, (0.85, 0.90, 0.93, 0.96))):
    rates = [sum(cuckoo_insert(int(L * n_bins), n_bins, k, r) for _ in range(20)) / 20 for L in loads]
    print(f"cuckoo, {k} hashes: " + ", ".join(f"load {L:.2f}: {x:.0%}" for L, x in zip(loads, rates)))
for ratio in (1.15, 1.20, 1.25, 1.30):
    ok = sum(two_core(1000, int(ratio * 1000), 3, r) == 0 for _ in range(20))
    print(f"peeling, 3 hashes, {ratio:.2f} n positions: success in {ok}/20 runs")

**Think:** Why can't such experiments certify a failure probability of $2^{-40}$?

## Homework — PSI-sum: the total excess income
Agency B's value for $y$ is `income[y]` (excess income in euros). Implement the protocol of *Private Join and Compute* (Session I) with the toy Paillier of the library (`paillier_keygen`, `paillier_enc`, `paillier_add`, `paillier_dec`):
the Hatter sends $\{(H(y)^\beta,\ \mathrm{Enc}(v(y)))\}$; Alice, who knows $\{H(x)^{\alpha\beta}\}$, adds the ciphertexts of the matches and returns one ciphertext; the Hatter decrypts the sum.

In [ ]:
income = excess_income(Y)

def psi_sum(X, Y, income):
    raise NotImplementedError  # TODO

In [ ]:
# ----- check -----
print("PSI-sum:", psi_sum(X, Y, income), " true:", sum(income[x] for x in set(X) & set(Y)))

## Extra (optional) — two ideas from Sessions II and III, ready to run
**XOR-OKVS** (Session II): hide key-value pairs in a vector $S$ of about $1.3n$ entries; decoding XORs three positions. **CLR17 in the clear** (Session III): $d_x=r\prod_{y\in Y}(x-y)$ mod a small prime $t$, after hashing the identifiers into $\mathbb Z_t$; the FHE version computes exactly this, under encryption.

In [ ]:
def xor_okvs(pairs, size, seed=7):
    pos = lambda key: random.Random(sha(str(seed) + key)).sample(range(size), 3)
    P = {k: pos(k) for k in pairs}
    inc = [[] for _ in range(size)]
    for k in pairs:
        for v in P[k]: inc[v].append(k)
    deg = [len(l) for l in inc]; alive = set(pairs); order = []
    stack = [v for v in range(size) if deg[v] == 1]
    while stack:                                          # peeling
        v = stack.pop()
        if deg[v] != 1: continue
        k = next(k for k in inc[v] if k in alive); alive.remove(k); order.append((k, v))
        for u in P[k]:
            deg[u] -= 1
            if deg[u] == 1: stack.append(u)
    if alive: return None, pos
    S = [rng.getrandbits(64) for _ in range(size)]
    for k, free in reversed(order):                       # assign in reverse order
        a, b, c = P[k]; S[free] = 0; S[free] = pairs[k] ^ S[a] ^ S[b] ^ S[c]
    return S, pos

kv = {x: rng.getrandbits(64) for x in X}
S, pos = xor_okvs(kv, 140)
if S is not None:
    print("XOR-OKVS: keys decoding correctly:", sum(S[a] ^ S[b] ^ S[c] == v for k, v in kv.items() for a, b, c in [pos(k)]), "of", len(kv))

t = 65537
hx = lambda s: int(sha(s), 16) % t
hits = 0
for x in X[:20]:
    d = rng.randrange(1, t)
    for y in Y: d = d * (hx(x) - hx(y)) % t
    hits += (d == 0)
print(f"CLR17 in the clear, 20 elements: {hits} hits; true matches {len(set(X[:20]) & set(Y))}")

**Think:** CLR17: which operations would run under FHE, how many multiplications, what depth? Where do extra hits come from (Session I, truncation bound)?